# 07 — Run via the High-Level API

A clean, minimal entry point. Configure `ActiveLearningConfig`, provide circuits and a ground truth, and let `ActiveLearningLoop` handle everything.

Edit the config cell, then run all cells.

In [ ]:
import os
# Navigate to repo root so all relative paths resolve correctly.
if os.path.basename(os.getcwd()) in ('notebooks', 'examples'):
    os.chdir('..')
import pickle
import numpy as np
from active_learning.config import ActiveLearningConfig
from active_learning.loop import ActiveLearningLoop
from gcad.circuit import Topo  # required for pickle deserialization

## 1. Configure

In [ ]:
config = ActiveLearningConfig(
    max_cycles=8,
    ensemble_size=100,
    spread_factor=2.0,
    dist_type='lognormal',
    budget_circuits=2,
    budget_dosages=2,
    dosages=list(np.round(np.arange(0.2, 4.2, 0.4), 2)),
    measurement_noise_std=5.0,
    perturbation_scale=0.1,
    selection_ratio=0.2
)

## 2. Load Inputs

In [ ]:
# Load circuits (from notebook 01 or your own GCAD results)
with open(os.path.join("data", "selected_M_circuits.pkl"), "rb") as f:
    circuits_list = pickle.load(f)

circuit_dict = {f"Circuit_{i+1}": c for i, c in enumerate(circuits_list)}
print(f"Loaded {len(circuit_dict)} circuits: {list(circuit_dict.keys())}")

# Load ground truth (from notebook 02, or provide your own true parameters)
with open(os.path.join("data", "ground_truth", "true_parts.pkl"), "rb") as f:
    true_parts = pickle.load(f)
print("Loaded ground truth parameters.")

## 3. Run

In [ ]:
np.random.seed(42)
engine = ActiveLearningLoop(circuit_dict, true_parts, config)
calibrated_parts, history = engine.run()

# Save calibrated parameters
os.makedirs("workspace", exist_ok=True)
with open("workspace/calibrated_parts.pkl", "wb") as f:
    pickle.dump(calibrated_parts, f)
print("\nSaved calibrated_parts.pkl.")